# Cosmo DL: frozen-fold TCN
Требуются код DL, принятый ML C-03 manifest и GPU. Notebook не создаёт folds и не меняет критерий оценки. Сначала preflight и CPU smoke; затем 3 seed на GPU.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil
REPO = Path('/kaggle/input/cosmo-dl/repo')
FOLD_MANIFEST = Path('/kaggle/input/cosmo-dl/inputs/dl_c03.json')
OUTPUT = Path('/kaggle/working/dl_tcn_run')
assert (REPO / 'src/veg_recovery/dl/train.py').is_file(), 'Укажите REPO с кодом DL'
assert FOLD_MANIFEST.is_file(), 'Дождитесь принятого C-03 от ML'
env = dict(os.environ, PYTHONPATH=str(REPO / 'src'), CUBLAS_WORKSPACE_CONFIG=':4096:8')
def run(*args):
    subprocess.run([sys.executable, *args], cwd=REPO, env=env, check=True)
run('-m', 'veg_recovery.dl.train', '--fold-manifest', str(FOLD_MANIFEST), '--preflight-only')

In [ ]:
import torch
assert torch.cuda.is_available(), 'Включите GPU в настройках Kaggle Notebook'
print('GPU:', torch.cuda.get_device_name(0), 'torch:', torch.__version__, 'CUDA:', torch.version.cuda)
run('-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'tests/dl', 'tests/anomalies')
run('-m', 'veg_recovery.dl.train', '--smoke', '--device', 'cuda', '--epochs', '3', '--window', '15', '--hidden-size', '16', '--layers', '2', '--output', '/kaggle/working/dl_gpu_smoke')

In [ ]:
run('-m', 'veg_recovery.dl.train', '--fold-manifest', str(FOLD_MANIFEST), '--device', 'cuda', '--seeds', '17', '42', '73', '--window', '61', '--epochs', '40', '--patience', '6', '--output', str(OUTPUT))
report = json.loads((OUTPUT / 'cv_report.json').read_text())
print(json.dumps(report, indent=2, ensure_ascii=False))
archive = shutil.make_archive(str(OUTPUT), 'zip', root_dir=OUTPUT)
print('Скачать:', archive)